# DRF Filtering, Search & Ordering

## Why Filtering?

Without filtering, a list endpoint always returns every object. With hundreds of thousands of rows that is slow and wasteful. Clients need to narrow results — by author, price range, date, or free text — without separate endpoints for each combination.

DRF supports three approaches:
- Manual queryset filtering inside `get_queryset()`
- `django-filter` integration for declarative filter classes
- DRF's built-in `SearchFilter` and `OrderingFilter` backends


## Manual Filtering in get_queryset()

Read query parameters from `self.request.query_params` and apply ORM filters:

```python
from rest_framework import viewsets
from .models import Book
from .serializers import BookSerializer

class BookViewSet(viewsets.ModelViewSet):
    serializer_class = BookSerializer

    def get_queryset(self):
        qs = Book.objects.all()

        author = self.request.query_params.get('author')
        if author:
            qs = qs.filter(author__slug=author)

        min_price = self.request.query_params.get('min_price')
        if min_price:
            qs = qs.filter(price__gte=min_price)

        return qs
```

Request: `GET /books/?author=tolkien&min_price=10`


## django-filter Integration

`django-filter` generates filters declaratively from field definitions.

```bash
pip install django-filter
```

```python
# settings.py
INSTALLED_APPS += ['django_filters']
```

### FilterSet class

```python
# books/filters.py
import django_filters
from .models import Book

class BookFilter(django_filters.FilterSet):
    min_price = django_filters.NumberFilter(field_name='price', lookup_expr='gte')
    max_price = django_filters.NumberFilter(field_name='price', lookup_expr='lte')
    author    = django_filters.CharFilter(field_name='author__slug', lookup_expr='iexact')
    title     = django_filters.CharFilter(lookup_expr='icontains')

    class Meta:
        model  = Book
        fields = ['min_price', 'max_price', 'author', 'title']
```

### Attach to a ViewSet

```python
from django_filters.rest_framework import DjangoFilterBackend
from .filters import BookFilter

class BookViewSet(viewsets.ModelViewSet):
    queryset         = Book.objects.all()
    serializer_class = BookSerializer
    filter_backends  = [DjangoFilterBackend]
    filterset_class  = BookFilter
```

Request: `GET /books/?min_price=5&max_price=30&title=python`


## SearchFilter

`SearchFilter` adds a single `?search=` parameter that performs a case-insensitive partial match across specified fields.

```python
from rest_framework.filters import SearchFilter

class BookViewSet(viewsets.ModelViewSet):
    queryset         = Book.objects.all()
    serializer_class = BookSerializer
    filter_backends  = [SearchFilter]
    search_fields    = ['title', 'author__name', 'description']
```

Request: `GET /books/?search=python`

### Field lookup prefixes

| Prefix | Behavior |
|--------|----------|
| (none) | `icontains` |
| `^` | `istartswith` |
| `=` | `iexact` |
| `@` | Full-text search (PostgreSQL) |
| `$` | Regex |

```python
search_fields = ['^title', '=isbn', 'author__name']
```


## OrderingFilter

`OrderingFilter` adds a `?ordering=` parameter. Clients can sort by any field listed in `ordering_fields`.

```python
from rest_framework.filters import OrderingFilter

class BookViewSet(viewsets.ModelViewSet):
    queryset         = Book.objects.all()
    serializer_class = BookSerializer
    filter_backends  = [OrderingFilter]
    ordering_fields  = ['title', 'price', 'published_at']
    ordering         = ['-published_at']   # default ordering
```

Examples:
```
GET /books/?ordering=price          # ascending
GET /books/?ordering=-price         # descending
GET /books/?ordering=author__name   # traverse relation
```


## Combining All Three Backends

```python
from django_filters.rest_framework import DjangoFilterBackend
from rest_framework.filters import SearchFilter, OrderingFilter
from .filters import BookFilter

class BookViewSet(viewsets.ModelViewSet):
    queryset         = Book.objects.select_related('author').all()
    serializer_class = BookSerializer
    filter_backends  = [DjangoFilterBackend, SearchFilter, OrderingFilter]
    filterset_class  = BookFilter
    search_fields    = ['title', 'author__name']
    ordering_fields  = ['title', 'price', 'published_at']
    ordering         = ['-published_at']
```

Request combining all three:
```
GET /books/?min_price=10&search=django&ordering=-price
```

DRF applies backends in order: filter → search → ordering.


## Testing Filters

```python
from rest_framework.test import APITestCase
from books.models import Author, Book

class BookFilterTests(APITestCase):
    def setUp(self):
        self.alice = Author.objects.create(name='Alice', slug='alice')
        Book.objects.create(title='Python Basics', author=self.alice, price=15)
        Book.objects.create(title='Advanced Python', author=self.alice, price=30)

    def test_min_price_filter(self):
        resp = self.client.get('/api/books/?min_price=20')
        self.assertEqual(resp.status_code, 200)
        self.assertEqual(len(resp.data['results']), 1)
        self.assertEqual(resp.data['results'][0]['title'], 'Advanced Python')

    def test_search(self):
        resp = self.client.get('/api/books/?search=basics')
        self.assertEqual(len(resp.data['results']), 1)

    def test_ordering(self):
        resp = self.client.get('/api/books/?ordering=price')
        prices = [r['price'] for r in resp.data['results']]
        self.assertEqual(prices, sorted(prices))
```


## Summary

- Filtering narrows query results without multiplying endpoints.
- Manual filtering via `get_queryset()` is flexible but verbose for many parameters.
- `django-filter` provides declarative `FilterSet` classes with typed filter fields and lookup expressions.
- `SearchFilter` adds a single `?search=` parameter for full-text-style matching across multiple fields.
- `OrderingFilter` exposes `?ordering=field` for ascending and `-field` for descending sorting.
- Combine backends by listing them in `filter_backends`; DRF applies them in sequence.
- Set `ordering` on the ViewSet to provide a default sort order when the client does not specify `?ordering=`.
